# 15 — Price co-movement

Estimate Portugal-Spain retail price co-movement using weekly pre-tax prices. The notebook requires a tidy price-history extraction and persists coefficients and diagnostics.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

import statsmodels.api as sm


## Required tidy input contract

Create `data/interim/weekly_oil_prices_tidy.csv` from the Commission workbook with:

`date, country, product, price_with_tax_eur_per_1000l, price_without_tax_eur_per_1000l`

for at least `PT` and `ES`, and products `diesel` and `gasoline`.

The source workbook is deliberately inventoried in notebook 05 because historical sheet layouts can change. Never silently guess columns if the workbook changes.


In [ ]:
path = PATHS.interim / "weekly_oil_prices_tidy.csv"
if not path.exists():
    raise FileNotFoundError(
        "Missing data/interim/weekly_oil_prices_tidy.csv. Extract it from the audited EC workbook before running price models."
    )
prices = pd.read_csv(path, parse_dates=["date"])
required = {"date", "country", "product", "price_without_tax_eur_per_1000l"}
if not required.issubset(prices.columns):
    raise ValueError(f"Price input missing: {sorted(required - set(prices.columns))}")


In [ ]:
# PT pre-tax price on Spain pre-tax price, with a post-transition interaction for changed co-movement.
rows = []
for product in ["diesel", "gasoline"]:
    sub = prices.loc[prices["product"] == product].copy()
    wide = sub.pivot(index="date", columns="country", values="price_without_tax_eur_per_1000l").dropna(subset=["PT", "ES"]).reset_index()
    wide["trend"] = np.arange(len(wide), dtype=float)
    cutoff = pd.Timestamp("2021-05-01")
    wide["post"] = (wide["date"] >= cutoff).astype(int)
    wide["ES_x_post"] = wide["ES"] * wide["post"]
    post_origin = np.searchsorted(wide["date"].to_numpy(), np.datetime64(cutoff))
    wide["post_trend"] = np.maximum(wide["trend"] - post_origin, 0.0)
    x = sm.add_constant(wide[["ES", "ES_x_post", "trend", "post", "post_trend"]], has_constant="add")
    model = sm.OLS(wide["PT"], x).fit(cov_type="HAC", cov_kwds={"maxlags": 8})
    for term in model.params.index:
        rows.append({"product": product, "term": term, "estimate": model.params[term], "std_error": model.bse[term], "p_value": model.pvalues[term], "nobs": model.nobs, "covariance": "HAC(8)", "outcome": "PT pre-tax EUR/1000L", "comparison": "ES pre-tax EUR/1000L", "model": "PT-ES price co-movement with ES_x_post interaction"})
coefs = pd.DataFrame(rows)
persist_dataframe(coefs, PATHS.metrics / "price_comovement_models.csv")
display(coefs)
